🔹 Step 1: Install Required Libraries


In [ ]:
%pip install transformers datasets torch
%pip install seqeval

In [4]:
from transformers import AutoModelForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from datasets import Dataset, Features, Sequence, ClassLabel, Value
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset, Features, Sequence, ClassLabel, Value

In [ ]:
#Declare label names
label_names = ["O", "B-AGE", "I-AGE", "B-GENDER", "I-GENDER", "B-ADDRESS", "I-ADDRESS", "B-SKILLS", "I-SKILLS", "B-EXPERIENCE", "I-EXPERIENCE", "B-EDUCATION", "I-EDUCATION", "B-CERTIFICATION", "I-CERTIFICATION",'B-Others', 'B-Role', 'I-Others', 'I-Role']


# Load the tokenizer and model
model_name = "distilroberta-base"
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),  # Set num_labels to 15
    ignore_mismatched_sizes=True,  # Ignore size mismatch in the classification layer
)

In [ ]:
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# Function to load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    tokens = []
    labels = []
    current_tokens = []
    current_labels = []

    for line in lines:
        if line.strip() == "":
            if current_tokens:
                tokens.append(current_tokens)
                labels.append(current_labels)
                current_tokens = []
                current_labels = []
        else:
            parts = line.strip().split()
            current_tokens.append(parts[0])
            current_labels.append(parts[-1])

    return {"tokens": tokens, "labels": labels}

# Load your dataset
train_data = load_dataset("/content/drive/MyDrive/conlls/combined-for-train.conll")
val_data = load_dataset("/content/drive/MyDrive/conlls/combined-for-eval.conll")

# Print the structure of train_data to verify
print("Train data structure:", train_data)

# Extract unique labels
def get_label_names(dataset):
    label_names = set()
    for labels in dataset["labels"]:  # Iterate over the "labels" key
        label_names.update(labels)    # Add all labels to the set
    return sorted(list(label_names))  # Convert to a sorted list

label_names = get_label_names(train_data)
print("Label names:", label_names)
print("Number of labels:", len(label_names))
# Define features
features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=label_names))
})

# Convert to datasets.Dataset
train_dataset = Dataset.from_dict(train_data, features=features)
val_dataset = Dataset.from_dict(val_data, features=features)

In [ ]:
def tokenize_and_align_labels(examples):
    # Tokenize the inputs
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,          # Enable truncation
        padding="max_length",     # Pad to the maximum length
        is_split_into_words=True, # Indicate that inputs are pre-tokenized
        max_length=512,           # Set a maximum length (adjust as needed)
        return_tensors="pt",      # Return PyTorch tensors
    )

    # Align labels with tokenized inputs
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective words
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens (e.g., [CLS], [SEP], padding) get a label of -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # New word, assign the corresponding label
                label_ids.append(label[word_idx])
            else:
                # Same word as previous token (subword tokenization), assign -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Add labels to the tokenized inputs
    tokenized_inputs["labels"] = labels

    # Return all necessary features, including input_ids and attention_mask
    return tokenized_inputs

# Tokenize and align labels for the train and validation datasets
train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
val_dataset = val_dataset.map(tokenize_and_align_labels, batched=True)

# Remove the "tokens" column as it's no longer needed
train_dataset = train_dataset.remove_columns(["tokens"])
val_dataset = val_dataset.remove_columns(["tokens"])

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/RoBERTa-fine-tuning-args/results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,  # Reduce batch size for low memory
    per_device_eval_batch_size=16,   # Reduce batch size for low memory
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="/content/drive/MyDrive/RoBERTa-fine-tuning-args/logs",
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    # fp16=False,  # Disable mixed precision if not using a compatible GPU
)

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Flatten the lists
    true_labels_flat = [label for sublist in true_labels for label in sublist]
    true_predictions_flat = [label for sublist in true_predictions for label in sublist]

    # Compute precision, recall, F1, and accuracy
    precision = precision_score(true_labels_flat, true_predictions_flat, average="weighted", zero_division=0)
    recall = recall_score(true_labels_flat, true_predictions_flat, average="weighted")
    f1 = f1_score(true_labels_flat, true_predictions_flat, average="weighted")
    accuracy = accuracy_score(true_labels_flat, true_predictions_flat)

    # Compute confusion matrix
    conf_matrix = confusion_matrix(true_labels_flat, true_predictions_flat, labels=label_names)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "confusion_matrix": conf_matrix.tolist(),  # Convert NumPy array to list
    }


In [ ]:
from transformers import Trainer

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  # Add the metrics function
)


In [ ]:
trainer.train()

In [ ]:
# Evaluate the model
results = trainer.evaluate()

# Print the evaluation results
print("Evaluation Results:")
print(f"Precision: {results['eval_precision']}")
print(f"Recall: {results['eval_recall']}")
print(f"F1 Score: {results['eval_f1']}")
print(f"Accuracy: {results['eval_accuracy']}")
print("Confusion Matrix:")
print(results["eval_confusion_matrix"])

In [ ]:
# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(results["eval_confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
model.save_pretrained("/content/drive/MyDrive/RoBERTa-fine-tuned-model")
tokenizer.save_pretrained("/content/drive/MyDrive/RoBERTa-fine-tuned-model")

In [ ]:
from transformers import pipeline

model
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)
results = ner_pipeline("contact wwwlinkedincominhazelwhitfield linkedin top skill responsive web design skilled multitasker web development certification lean six sigma yellow belt certification full stack web mobile development lean six sigma dmaic certification lean six sigma project management certification honorsawards honor certificate front end web mobile developmenthazel whitfield accounting professional greater orlando experience careersource central florida senior accounting specialist september two thousand twentyfour present 6 month ra simasek pa office manager bookkeeper may two thousand twentyfour july two thousand twentyfour 3 month careersource central florida three year four month senior accounting specialist july two thousand twentyone november two thousand twentytwo 1 year five month accounting specialist august two thousand nineteen july two thousand twentyone 2 year kermali cpa pa staff accountant april two thousand seventeen july two thousand nineteen 2 year four month stewart company staff accountant december two thousand fifteen april two thousand seventeen 1 year five month education aclc college office information system business office administration service 2011 2013 page one one")
print(results)